# T2 - Classifier Inference Entry Point

**Guardian Eye task:** T2

Run this notebook top-to-bottom on the target virtual machine. Configuration is centralized near the top, and heavy model work only occurs in explicitly marked execution cells.


## 1. Imports and stable output schema

Defines the JSON-compatible contract consumed by the evidence builder, narrator, and API.


In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Mapping, Sequence
import gc
import importlib
import json
import os
import numpy as np

@dataclass
class ClassifierResult:
    verdict: str
    confidence: float
    threshold: float
    gate: list[float]
    gqs: list[float]
    telemetry: dict[str, Any]
    frames_vit: np.ndarray
    source: dict[str, Any]
    timestamp: str

    def json_metadata(self) -> dict[str, Any]:
        return {
            "verdict": self.verdict,
            "confidence": self.confidence,
            "threshold": self.threshold,
            "gate": self.gate,
            "gqs": self.gqs,
            "telemetry": to_jsonable(self.telemetry),
            "source": to_jsonable(self.source),
            "timestamp": self.timestamp,
            "frames_shape": list(self.frames_vit.shape),
            "frames_dtype": str(self.frames_vit.dtype),
        }

def to_jsonable(value: Any) -> Any:
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, Mapping):
        return {str(k): to_jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(v) for v in value]
    return value


## 2. Dynamic adapter loader

The existing V9 implementation is not duplicated here. Configure an adapter as `module.path:function_name`; that function must load and return the classifier pipeline.


In [ ]:
def import_callable(spec: str) -> Callable[..., Any]:
    module_name, function_name = spec.split(":", maxsplit=1)
    module = importlib.import_module(module_name)
    function = getattr(module, function_name)
    if not callable(function):
        raise TypeError(f"{spec} is not callable")
    return function

def load_v9_pipeline(adapter_spec: str | None = None) -> Any:
    spec = adapter_spec or os.getenv(
        "CLASSIFIER_ADAPTER", "models.classifier.v9_adapter:load_classifier"
    )
    return import_callable(spec)(model_path=os.getenv("CLASSIFIER_MODEL_PATH"))


## 3. Normalize and validate V9 output

Converts the existing pipeline output into the stable contract and rejects malformed results early.


In [ ]:
def normalize_v9_output(raw: Mapping[str, Any], video_path: str | Path) -> ClassifierResult:
    frames = np.asarray(raw["frames_vit"])
    result = ClassifierResult(
        verdict=str(raw["verdict"]),
        confidence=float(raw["confidence"]),
        threshold=float(raw["threshold"]),
        gate=np.asarray(raw["gate"], dtype=float).reshape(-1).tolist(),
        gqs=np.asarray(raw["gqs"], dtype=float).reshape(-1).tolist(),
        telemetry=dict(raw["telemetry"]),
        frames_vit=frames,
        source=dict(raw.get("source") or {"video_path": str(Path(video_path).resolve())}),
        timestamp=str(raw.get("timestamp") or datetime.now(timezone.utc).isoformat()),
    )
    validate_classifier_result(result)
    return result

def validate_classifier_result(result: ClassifierResult) -> None:
    if result.verdict not in {"violence", "non-violence"}:
        raise ValueError("verdict must be 'violence' or 'non-violence'")
    if not 0.0 <= result.confidence <= 1.0:
        raise ValueError("confidence must be between 0 and 1")
    if len(result.gate) != 4:
        raise ValueError(f"gate must contain 4 values, got {len(result.gate)}")
    if len(result.gqs) != 5:
        raise ValueError(f"gqs must contain 5 values, got {len(result.gqs)}")
    if result.frames_vit.shape != (16, 224, 224, 3):
        raise ValueError(f"frames_vit must have shape (16,224,224,3), got {result.frames_vit.shape}")
    if not np.isfinite(result.confidence):
        raise ValueError("confidence must be finite")


## 4. Explicit classifier cleanup

Releases references, Python garbage, and unused CUDA cache before the narrator stage is allowed to load.


In [ ]:
def release_classifier(pipeline: Any) -> None:
    unload = getattr(pipeline, "unload", None)
    if callable(unload):
        unload()
    del pipeline
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except ImportError:
        pass


## 5. Public inference entry point

`analyze_clip_core` is the single classifier-facing function. It always attempts cleanup, even when inference fails.


In [ ]:
def analyze_clip_core(
    video_path: str | Path,
    *,
    adapter_spec: str | None = None,
) -> ClassifierResult:
    video_path = Path(video_path)
    if not video_path.is_file():
        raise FileNotFoundError(video_path)

    pipeline = None
    try:
        pipeline = load_v9_pipeline(adapter_spec)
        if hasattr(pipeline, "analyze_clip"):
            raw = pipeline.analyze_clip(str(video_path))
        elif callable(pipeline):
            raw = pipeline(str(video_path))
        else:
            raise TypeError("V9 adapter must return a callable or an object with analyze_clip()")
        return normalize_v9_output(raw, video_path)
    finally:
        if pipeline is not None:
            release_classifier(pipeline)


## 6. Optional artifact persistence

Stores metadata as JSON and frame tensors as compressed NPZ. This avoids putting large frame arrays inside API JSON.


In [ ]:
def save_classifier_artifacts(result: ClassifierResult, output_dir: str | Path) -> dict[str, str]:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    stem = Path(result.source.get("video_path", "clip")).stem
    metadata_path = output_dir / f"{stem}_classifier.json"
    frames_path = output_dir / f"{stem}_frames_vit.npz"
    metadata_path.write_text(json.dumps(result.json_metadata(), indent=2), encoding="utf-8")
    np.savez_compressed(frames_path, frames_vit=result.frames_vit)
    return {"metadata_path": str(metadata_path), "frames_path": str(frames_path)}

# Target-VM execution example:
# result = analyze_clip_core("data/incoming/example.mp4")
# paths = save_classifier_artifacts(result, "data/telemetry")
